In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)


project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.serving import load_model

print("Project Root:", project_root)

Project Root: /home/pasali/Desktop/Data/AI-Engineer/Hafta-2/ShopperIntent


In [2]:
data_path = project_root / "data" / "online_shoppers_intention.csv"

df = pd.read_csv(data_path)


X = df.drop(columns=["Revenue", "PageValues"])
y = df["Revenue"].astype(int)


X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (8631, 16)
Validation: (1849, 16)
Test: (1850, 16)


In [3]:
model, preprocessor, config = load_model()

threshold = config["decision_threshold"]

assert threshold == 0.60

assert X_test.columns.tolist() == config["feature_columns"]

print("Model:", config["model_name"])
print("Input Dimension:", config["input_dim"])
print("Threshold:", threshold)
print("Model Training Mode:", model.training)

Model: ShopperIntent
Input Dimension: 73
Threshold: 0.6
Model Training Mode: False


In [4]:
X_val_processed = preprocessor.transform(
    X_val[config["feature_columns"]]
)

if hasattr(X_val_processed, "toarray"):
    X_val_processed = X_val_processed.toarray()

X_val_tensor = torch.tensor(
    np.asarray(X_val_processed, dtype=np.float32)
)

model.eval()

with torch.no_grad():

    val_logits = model(X_val_tensor)

    val_probabilities = torch.softmax(
        val_logits,
        dim=1
    )[:, config["positive_class"]].numpy()

val_predictions = (
    val_probabilities >= threshold
).astype(int)

tn_val, fp_val, fn_val, tp_val = confusion_matrix(
    y_val,
    val_predictions,
    labels=[0, 1]
).ravel()

print("TP:", tp_val)
print("FP:", fp_val)
print("TN:", tn_val)
print("FN:", fn_val)

# Önceki validation sonuçlarımız
assert (tp_val, fp_val, tn_val, fn_val) == (
    30, 40, 1523, 256
), "Kaydedilmiş model önceki validation sonuçlarıyla eşleşmiyor!"

print("Validation doğrulaması başarılı!")

TP: 30
FP: 40
TN: 1523
FN: 256
Validation doğrulaması başarılı!


In [5]:
X_test_processed = preprocessor.transform(
    X_test[config["feature_columns"]]
)

if hasattr(X_test_processed, "toarray"):
    X_test_processed = X_test_processed.toarray()

X_test_tensor = torch.tensor(
    np.asarray(X_test_processed, dtype=np.float32)
)

model.eval()

with torch.no_grad():


    logits = model(X_test_tensor)


    positive_probabilities = torch.softmax(
        logits,
        dim=1
    )[:, config["positive_class"]].numpy()


predictions = (
    positive_probabilities >= threshold
).astype(int)

print("Test kullanıcı sayısı:", len(predictions))
print("İlk 10 olasılık:", positive_probabilities[:10])

Test kullanıcı sayısı: 1850
İlk 10 olasılık: [0.0270964  0.25797457 0.03965422 0.01294555 0.1210648  0.00268897
 0.01948245 0.23080401 0.4084835  0.03981481]


In [6]:
accuracy = accuracy_score(y_test, predictions)

precision = precision_score(
    y_test,
    predictions,
    zero_division=0
)

recall = recall_score(
    y_test,
    predictions,
    zero_division=0
)

f1 = f1_score(
    y_test,
    predictions,
    zero_division=0
)

ap = average_precision_score(
    y_test,
    positive_probabilities
)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    predictions,
    labels=[0, 1]
).ravel()

test_results = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "average_precision": ap,
    "tp": tp,
    "fp": fp,
    "tn": tn,
    "fn": fn
}

test_df = pd.DataFrame([test_results])

print(test_df.to_string(index=False))

 accuracy  precision   recall       f1  average_precision  tp  fp   tn  fn
 0.845946   0.507246 0.122378 0.197183           0.345782  35  34 1530 251


In [7]:
validation_results = {
    "accuracy": 0.8399,
    "precision": 0.4286,
    "recall": 0.1049,
    "f1": 0.1685
}

comparison_df = pd.DataFrame({
    "Validation": validation_results,
    "Test": {
        metric: test_results[metric]
        for metric in validation_results
    }
})

print(comparison_df.round(4))

           Validation    Test
accuracy       0.8399  0.8459
precision      0.4286  0.5072
recall         0.1049  0.1224
f1             0.1685  0.1972


In [8]:
reports_dir = project_root / "reports"

reports_dir.mkdir(parents=True, exist_ok=True)

csv_path = reports_dir / "final_test_results.csv"

test_df.to_csv(
    csv_path,
    index=False
)

print("CSV kaydedildi:", csv_path)
print("Dosya mevcut mu?", csv_path.is_file())

CSV kaydedildi: /home/pasali/Desktop/Data/AI-Engineer/Hafta-2/ShopperIntent/reports/final_test_results.csv
Dosya mevcut mu? True
